# Router experiments — train, validate, use

Minimal harness for testing configs. Edit the **config** cell, then Run All.
Every knob is an existing `RouterExperiment` / `StrategyRouter` flag.

In [2]:
%load_ext autoreload
%autoreload 2

In [1]:
from hybrid_search_rrf_dataset.router import (
    QueryEncoder,
    Representation,
    RouterExperiment,
    StrategyRouter,
)

# ---- config: edit and Run All ----
REPRESENTATION = Representation.ENGINEERED  # ENGINEERED | EMBEDDING | BOTH
ENCODER = None           # QueryEncoder() for EMBEDDING/BOTH (downloads e5 once)
PROTOCOL = 'random_within_lane'  # or 'holdout_lane' (train/use cells only)
ALL_ROWS = 'decisive'     # 'decisive' ~4K clear wins | 'recommended' ~12K all real winners | 'all' ~37K, tied/zero rows as negatives
MAX_CLASS_SHARE = None    # e.g. 0.5: downsample so no winner class exceeds that share

/Users/andrei/projects/hybrid-search-rrf-dataset/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [34]:
# 1 — train
from hybrid_search_rrf_dataset.router import _derive_engineered, _margin, _routes_differ

exp = RouterExperiment(encoder=ENCODER)
train, test = exp.split(PROTOCOL)

router = StrategyRouter(REPRESENTATION, encoder=ENCODER).fit(
    train, all_rows=ALL_ROWS, max_class_share=MAX_CLASS_SHARE
)
# router.tune_thresholds(train)
print('thresholds (dense, sparse):', tuple(round(t, 3) for t in router.thresholds))

# coefficients + support: `fires` = substrate rows where the feature is nonzero.
# A big weight with tiny support is a scaling artifact, not evidence.
mode = {False: 'decisive', True: 'recommended'}.get(ALL_ROWS, ALL_ROWS)
fit_rows = (
    train if mode == 'all'
    else train[_routes_differ(train)] if mode == 'recommended'
    else train[_margin(train) >= router.decisive_margin]
)
eng = _derive_engineered(fit_rows)
coef = router.coefficients().set_index('feature')
coef['fires'] = [(eng[f] > 0).sum() if f in eng.columns else None for f in coef.index]
coef.round(3).nlargest(8, 'sparse_weight')

thresholds (dense, sparse): (0.5, 0.5)


,dense_weight,sparse_weight,fires
feature,,,
length.length_chars,-0.891,0.945,4307
structured_identifiers.bic,-0.631,0.542,431
structured_identifiers.file_path,-0.561,0.411,29
sentence_markers.acronym,-0.441,0.251,634
syntactic_depth.nesting_depth,-0.292,0.237,4246
morphology.word_variation_share,-0.188,0.209,3634
logical_structures.temporal,-0.154,0.155,216
coordination.widest_list_size,-0.090,0.119,1386


In [35]:
# 2 — validate: six-column table, both protocols, this config
cols = [
    'protocol', 'representation', 'all_rows', 'max_class_share', 'n_test_decisive',
    'const_dense_only', 'const_pure_rrf', 'const_sparse_only',
    'oracle', 'router', 'headroom_captured', 't_dense', 't_sparse',
]
result = exp.run(
    representations=[REPRESENTATION],
    all_rows=ALL_ROWS,
    max_class_share=MAX_CLASS_SHARE,
)
result[cols].round(3)

holdout_lane·engineered: 100%|██████████| 2/2 [00:00<00:00,  4.64it/s, headroom=0.052, n=286]       


,protocol,representation,all_rows,max_class_share,n_test_decisive,const_dense_only,const_pure_rrf,const_sparse_only,oracle,router,headroom_captured,t_dense,t_sparse
0,random_within_lane,engineered,decisive,None,1082,0.625,0.219,0.371,0.977,0.693,0.193,0.20,0.80
1,holdout_lane,engineered,decisive,None,286,0.566,0.203,0.456,1.000,0.589,0.052,0.15,0.75


In [36]:
# 3 — use: route your own queries with the §1 router (needs en_core_web_sm)
my_queries = [
    'Who likes Curling?',
    'what are the side effects of DHA',
    'CVE-2021-44228 log4j remote code execution',
    'http://localhost.com',
]
for q in my_queries:
    e = router.explain(q)
    print(f"{e['route']!s:12s} p_dense={e['p_dense']:.2f} p_sparse={e['p_sparse']:.2f}  {q}")

dense_only   p_dense=0.59 p_sparse=0.42  Who likes Curling?
dense_only   p_dense=0.62 p_sparse=0.36  what are the side effects of DHA
sparse_only  p_dense=0.42 p_sparse=0.59  CVE-2021-44228 log4j remote code execution
sparse_only  p_dense=0.33 p_sparse=0.65  http://localhost.com


In [41]:
# 4 — serve: refit on ALL labelled data with this config (no held-out split).
# This is the model you'd ship; its numbers are NOT comparable to §2's,
# which must hold data out to stay an honest measurement.
data = exp.load()
served = StrategyRouter(REPRESENTATION, encoder=ENCODER, delta=0.05).fit(
    data, all_rows='all', max_class_share=MAX_CLASS_SHARE
)
served.tune_thresholds(data)
print('serving thresholds (dense, sparse):', tuple(round(t, 3) for t in served.thresholds))
for q in my_queries:
    e = served.explain(q)
    print(f"{e['route']!s:12s} p_dense={e['p_dense']:.2f} p_sparse={e['p_sparse']:.2f}  {q}")

serving thresholds (dense, sparse): (0.3, 0.5)
dense_only   p_dense=0.52 p_sparse=0.44  Who likes Curling?
dense_only   p_dense=0.54 p_sparse=0.38  what are the side effects of DHA
pure_rrf     p_dense=0.49 p_sparse=0.50  CVE-2021-44228 log4j remote code execution
pure_rrf     p_dense=0.55 p_sparse=0.54  http://localhost.com


In [42]:
# probe the served model — edit the list and rerun
probes = [
    'qdrant_client.http.models.FormulaQuery',
    'qdrant HNSW ef_construct default value',
    'error code 429 rate limit exceeded API',
    'docker-compose.yml volume mount permissions',
    'RTX 4090 vs A100 fp16 throughput benchmark',
    'python list comprehension syntax',
    'How can I make repeated code easier to maintain in Python?',
    'Why would a website tell me a request is malformed instead of unauthorized?',
    'dylans article about qdrant search',
    'qdrant search',
    '/dylan/neil/jenny',
    'ThinkPad X1 broken screen replacement',
    'ThindkPadX1 is genuinely great laptip',
]
for q in probes:
    e = served.explain(q)
    print(f"{e['route']!s:12s} p_dense={e['p_dense']:.2f} "
          f"p_sparse={e['p_sparse']:.2f}  {q}")


sparse_only  p_dense=0.41 p_sparse=0.55  qdrant_client.http.models.FormulaQuery
sparse_only  p_dense=0.38 p_sparse=0.60  qdrant HNSW ef_construct default value
sparse_only  p_dense=0.48 p_sparse=0.56  error code 429 rate limit exceeded API
sparse_only  p_dense=0.32 p_sparse=0.63  docker-compose.yml volume mount permissions
sparse_only  p_dense=0.42 p_sparse=0.60  RTX 4090 vs A100 fp16 throughput benchmark
sparse_only  p_dense=0.46 p_sparse=0.52  python list comprehension syntax
dense_only   p_dense=0.59 p_sparse=0.38  How can I make repeated code easier to maintain in Python?
dense_only   p_dense=0.52 p_sparse=0.41  Why would a website tell me a request is malformed instead of unauthorized?
dense_only   p_dense=0.38 p_sparse=0.49  dylans article about qdrant search
sparse_only  p_dense=0.38 p_sparse=0.55  qdrant search
dense_only   p_dense=0.59 p_sparse=0.50  /dylan/neil/jenny
sparse_only  p_dense=0.44 p_sparse=0.55  ThinkPad X1 broken screen replacement
sparse_only  p_dense=0.34 p_spa

In [43]:
golden_set_auto_fusion = [
    {"query": "who founded apple?", "expected_hi": 2, "expected_lo": 0},
    {"query": "how does photosynthesis work in plants", "expected_hi": 2, "expected_lo": 0},
    {"query": "explain quicksort", "expected_hi": 2, "expected_lo": 0},
    {"query": "best laptop for college students 2024", "expected_hi": 3, "expected_lo": 0},
    {"query": "comment volent les oiseaux", "expected_hi": 2, "expected_lo": 0},
    {"query": "como aprender a programar en rust", "expected_hi": 2, "expected_lo": 0},
    {"query": "tell me about dogs", "expected_hi": 2, "expected_lo": 0},
    {"query": "hey can you help me out", "expected_hi": 2, "expected_lo": 0},
    {"query": "550e8400-e29b-41d4-a716-446655440000", "expected_hi": 9, "expected_lo": 8},
    {"query": "00000000-0000-0000-0000-000000000000", "expected_hi": 9, "expected_lo": 8},
    {"query": "ERR_CONNECTION_RESET", "expected_hi": 9, "expected_lo": 8},
    {"query": "ENOENT", "expected_hi": 9, "expected_lo": 7},
    {"query": "HTTP 502", "expected_hi": 9, "expected_lo": 6},
    {"query": "v1.2.3 changelog", "expected_hi": 9, "expected_lo": 6},
    {"query": "Python 3.11.4 release notes", "expected_hi": 8, "expected_lo": 5},
    {"query": "a3f5d8b9e12c4d56789abcdef0123456", "expected_hi": 9, "expected_lo": 8},
    {"query": "/etc/nginx/nginx.conf", "expected_hi": 9, "expected_lo": 7},
    {"query": "B07XJ8C8F5", "expected_hi": 9, "expected_lo": 7},
    {"query": "GPT-3", "expected_hi": 8, "expected_lo": 5},
    {"query": "BERT model paper", "expected_hi": 7, "expected_lo": 4},
    {"query": "ThinkPad X1 broken screen replacement", "expected_hi": 6, "expected_lo": 3},
    {"query": "iPhone 15 Pro Max battery life", "expected_hi": 6, "expected_lo": 3},
    {"query": "kubernetes pod CrashLoopBackOff", "expected_hi": 8, "expected_lo": 5},
    {"query": "why does my Java program throw NullPointerException at line 42", "expected_hi": 6, "expected_lo": 3},
    {"query": "C++ undefined reference to vtable", "expected_hi": 8, "expected_lo": 5},
    {"query": "how to fix broken screen on my Lenovo ThinkPad X1", "expected_hi": 5, "expected_lo": 2},
    {"query": "linux", "expected_hi": 3, "expected_lo": 1},
    {"query": "covid", "expected_hi": 3, "expected_lo": 1},
    {"query": "the", "expected_hi": 2, "expected_lo": 0},
    {"query": "hello", "expected_hi": 2, "expected_lo": 0},
]

In [44]:
# router vs the production auto-fusion bands (0-2 dense, 3-6 rrf, 7-9 sparse):
# agree = the router's route falls inside the band the classifier would allow
import pandas as pd

from hybrid_search_rrf_dataset.router import _production_route

rows = []
for case in golden_set_auto_fusion:
    band = {_production_route(s)
            for s in range(case["expected_lo"], case["expected_hi"] + 1)}
    router_result = served.explain(case["query"])
    route = router_result["route"]
    rows.append({
        "query": case["query"],
        "router": route.value,
        "router_values": (router_result['p_dense'], router_result['p_sparse']),
        "autofision": (case["expected_lo"], case["expected_hi"]),
        "autofusion band": "..".join(sorted(r.value for r in band)),
        "agree": route in band,
    })
frame = pd.DataFrame(rows)
print(f"router inside the auto-fusion band on {frame['agree'].sum()}/{len(frame)}")
frame[~frame["agree"]]


router inside the auto-fusion band on 23/30


,query,router,router_values,autofision,autofusion band,agree
2,explain quicksort,sparse_only,"(0.42190366758392106, 0.5153792095774334)","(0, 2)",dense_only,False
10,ERR_CONNECTION_RESET,dense_only,"(0.39407013385897727, 0.49992302536321764)","(8, 9)",sparse_only,False
12,HTTP 502,dense_only,"(0.6462980686690533, 0.47677326420354854)","(6, 9)",pure_rrf..sparse_only,False
19,BERT model paper,dense_only,"(0.5782569050529998, 0.5138847027314318)","(4, 7)",pure_rrf..sparse_only,False
20,ThinkPad X1 broken screen replacement,sparse_only,"(0.4370490131391394, 0.5489888884863441)","(3, 6)",pure_rrf,False
21,iPhone 15 Pro Max battery life,dense_only,"(0.5887935394769154, 0.4838120524347817)","(3, 6)",pure_rrf,False
24,C++ undefined reference to vtable,dense_only,"(0.40662021629954515, 0.4937049577738079)","(5, 8)",pure_rrf..sparse_only,False


In [45]:
# 5 — margin hedge: rrf when |p_dense − p_sparse| < delta.
# Delta is tuned on the train routes_differ frame (never on test),
# then judged on held-out decisive rows against both baselines.
# If the best delta is 0.0, the data says the hedge doesn't pay.
import numpy as np

from hybrid_search_rrf_dataset.router import _mean_objective, _route_from_probs

tune_frame = train[_routes_differ(train)]
p_d, p_s = router._probabilities(tune_frame)
deltas = np.round(np.arange(0.0, 0.32, 0.002), 2)[::-1]
tune_scores = [
    _mean_objective(tune_frame, _route_from_probs(p_d, p_s, 0.5, 0.5, delta=d))
    for d in deltas
]
best_delta = float(deltas[int(np.argmax(tune_scores))])
print(f'best delta on train: {best_delta:.2f} '
      f'(objective {max(tune_scores):.4f} vs {tune_scores[-1]:.4f} at delta=0)')

decisive_test = test[_margin(test) >= router.decisive_margin]
pt_d, pt_s = router._probabilities(decisive_test)
configs = {
    'tuned thresholds, delta=0': (*router.thresholds, 0.0),
    'symmetric (0.5, 0.5), delta=0': (0.5, 0.5, 0.0),
    f'symmetric + delta={best_delta:.2f}': (0.5, 0.5, best_delta),
}
for label, (td, ts, d) in configs.items():
    routes = _route_from_probs(pt_d, pt_s, td, ts, delta=d)
    score = _mean_objective(decisive_test, routes)
    n_rrf = sum(r.value == 'pure_rrf' for r in routes)
    print(f'{label:32s} held-out: {score:.3f}  (rrf fired on {n_rrf}/{len(routes)})')


best delta on train: 0.00 (objective 0.4335 vs 0.4335 at delta=0)
tuned thresholds, delta=0        held-out: 0.671  (rrf fired on 9/1082)
symmetric (0.5, 0.5), delta=0    held-out: 0.671  (rrf fired on 9/1082)
symmetric + delta=0.00           held-out: 0.671  (rrf fired on 9/1082)


# 6 — Acceptability heads vs argmax router (SPEC d60)

The d60 form: three binary heads trained on `ok_* = score >= oracle − 0.3`
(hit parity), serving the cheapest route whose P(ok) clears the threshold.
Trains on **answerable** rows only — all_zero rows carry null labels — so
the 15K tied rows finally contribute (as sparse/dense/rrf positives) instead
of being filtered out as winnerless.

Read the table against the `serve oracle (view)` row — the cost-aware
ceiling. Two things to watch:

- **objective columns**: the heads router must hold the argmax router's
  quality (the objective ignores cost, so ties score identically whichever
  route serves them).
- **mean cost + route mix**: this is where the two forms should actually
  differ — the heads router is *trained* to prefer cheap-when-tied, the
  argmax router only ever saw winners.

In [46]:
import pandas as pd

from hybrid_search_rrf_dataset.labels import AcceptabilityLabels
from hybrid_search_rrf_dataset.router import AcceptabilityRouter

from hybrid_search_rrf_dataset.fusion import SERVING_COST, StrategyName

# same split, same representation as §1 — the label form is the only variable
acc = AcceptabilityRouter(
      REPRESENTATION, 
      encoder=ENCODER,
).fit(train)
print(f"tolerance {acc.tolerance} (hit parity) | threshold {acc.threshold}")

view = AcceptabilityLabels(test).frame()
answerable = view[view["serve"].notna()].reset_index(drop=True)
print(f"test: {len(answerable):,} answerable of {len(view):,} "
      f"({(view['serve'].isna()).sum():,} all_zero excluded)")

acc_routes = acc.predict_routes(answerable)
argmax_routes = router.predict_routes(answerable)   # the §1 router

tolerance 0.3 (hit parity) | threshold 0.5
test: 8,206 answerable of 10,322 (2,116 all_zero excluded)


In [47]:
import numpy as np

from hybrid_search_rrf_dataset.fusion import SERVING_COST, StrategyName
from hybrid_search_rrf_dataset.router import _margin, _mean_objective

decisive_mask = (_margin(answerable) >= router.decisive_margin).to_numpy()
serve_oracle = [StrategyName(s) for s in answerable["serve"]]


def readout(policy: str, routes) -> dict:
    routes = list(routes)
    on_decisive = [r for r, m in zip(routes, decisive_mask) if m]
    mix = pd.Series([r.value for r in routes]).value_counts(normalize=True)
    return {
        "policy": policy,
        "objective (answerable)": _mean_objective(answerable, routes),
        "objective (decisive)": _mean_objective(
            answerable[decisive_mask], on_decisive
        ),
        "serve agreement": float(np.mean(
            [r == s for r, s in zip(routes, serve_oracle)]
        )),
        "mean cost": float(np.mean([SERVING_COST[r] for r in routes])),
        **{f"% {s.value}": float(mix.get(s.value, 0.0)) for s in StrategyName},
    }


table = pd.DataFrame([
    readout("serve oracle (view)", serve_oracle),
    readout("acceptability heads", acc_routes),
    readout("argmax router (§1)", argmax_routes),
    *(readout(f"const {s.value}", [s] * len(answerable)) for s in StrategyName),
])
table.round(3)

,policy,objective (answerable),objective (decisive),serve agreement,mean cost,% dense_only,% pure_rrf,% sparse_only
0,serve oracle (view),0.782,0.977,1.000,0.228,0.213,0.007,0.779
1,acceptability heads,0.698,0.661,0.457,0.676,0.612,0.032,0.356
2,argmax router (§1),0.699,0.671,0.456,0.668,0.632,0.018,0.350
3,const dense_only,0.680,0.625,0.213,1.000,1.000,0.000,0.000
4,const pure_rrf,0.688,0.219,0.007,2.000,0.000,1.000,0.000
5,const sparse_only,0.587,0.371,0.779,0.000,0.000,0.000,1.000


In [48]:
# 6b — the cost policy lives in the inference rule, not the label (d60e).
# Same heads, same weights, no refit: only the serve rule changes.
#   priority=None        -> serve the most probable route (no cost policy)
#   priority=COST_ORDER  -> serve the first head clearing `threshold`, cheapest first
policies = {
    "heads, priority=None (most probable)": None,
    "heads, priority=COST_ORDER (cheapest)": AcceptabilityRouter.COST_ORDER,
}

rows = [readout("serve oracle (view)", serve_oracle)]
for label, priority in policies.items():
    acc.priority = priority
    rows.append(readout(label, acc.predict_routes(answerable)))
rows.append(readout("argmax router (§1)", argmax_routes))

acc.priority = None  # leave the corrected default in place for §7
pd.DataFrame(rows).round(3)

,policy,objective (answerable),objective (decisive),serve agreement,mean cost,% dense_only,% pure_rrf,% sparse_only
0,serve oracle (view),0.782,0.977,1.000,0.228,0.213,0.007,0.779
1,"heads, priority=None (most probable)",0.698,0.661,0.457,0.676,0.612,0.032,0.356
2,"heads, priority=COST_ORDER (cheapest)",0.684,0.637,0.543,0.526,0.489,0.018,0.493
3,argmax router (§1),0.699,0.671,0.456,0.668,0.632,0.018,0.350


In [49]:
# the serving surface, probed: three P(ok) per query, cheapest clearing wins
for q in my_queries:
    e = acc.explain(q)
    print(f"{e['route']!s:12s} "
          f"ok_dense={e['p_ok_dense_only']:.2f} "
          f"ok_rrf={e['p_ok_pure_rrf']:.2f} "
          f"ok_sparse={e['p_ok_sparse_only']:.2f}  {q}")

dense_only   ok_dense=0.59 ok_rrf=0.52 ok_sparse=0.49  Who likes Curling?
dense_only   ok_dense=0.61 ok_rrf=0.52 ok_sparse=0.47  what are the side effects of DHA
dense_only   ok_dense=1.00 ok_rrf=1.00 ok_sparse=0.00  CVE-2021-44228 log4j remote code execution
pure_rrf     ok_dense=0.46 ok_rrf=0.48 ok_sparse=0.43  http://localhost.com


# 7 — Archetype probes: where the router still fails

Seven queries whose route follows from retrieval mechanics alone, frozen from
the auto-fusion disagreement table above so a fix is checked against a fixed
instrument rather than an aggregate that hides it — the hex digest is roughly
0.1% of any eval slice. `HTTP 502` carries no expected route: its cell
predicts both, so only the corpus decides.

Two of these failures sit upstream of any model, and no amount of training
data reaches them:

- `natural_language_share` reads **1.0** for the bare hex digest. spaCy tags
  the unknown token `AUX`, a closed class, so the strongest natural-language
  signal in the set belongs to the query with the least natural language in
  it. Same failure d56 fixed for `NUM`, resurfacing through a different tag.
- `bare_concept_token` claims that digest, a bare error code, and a status
  code alike. Its predicate is only `length_words < 3` AND `number < 1`,
  while its own `looks_like` says opaque jargon belongs to the identifier
  cells — and the predicate is the sole matcher. Every short opaque token not
  claimed as a NUMBER lands in the archetype whose prior is `dense_only`.

Each probe's reason is on `ARCHETYPE_PROBES[i].why`.

In [50]:
from hybrid_search_rrf_dataset.probes import ARCHETYPE_PROBES, probe

argmax_probe, heads_probe = probe(served), probe(acc)
compare = argmax_probe[["query", "expected"]].copy()
compare["argmax route"] = argmax_probe["served"]
compare["argmax ok"] = argmax_probe["agrees"]
compare["heads route"] = heads_probe["served"]
compare["heads ok"] = heads_probe["agrees"]

for name, frame in (("argmax", argmax_probe), ("heads", heads_probe)):
    decided = frame["agrees"].notna().sum()
    print(f"{name:7s} agrees with mechanics on {frame['agrees'].eq(True).sum()}/{decided}")
compare

argmax  agrees with mechanics on 4/6
heads   agrees with mechanics on 3/6


,query,expected,argmax route,argmax ok,heads route,heads ok
0,a3f5d8b9e12c4d56789abcdef0123456,sparse_only,sparse_only,True,pure_rrf,False
1,/etc/nginx/nginx.conf,sparse_only,sparse_only,True,sparse_only,True
2,ERR_CONNECTION_RESET,sparse_only,dense_only,False,pure_rrf,False
3,explain quicksort,dense_only,sparse_only,False,sparse_only,False
4,HTTP 502,NaN,dense_only,<NA>,dense_only,<NA>
5,comment volent les oiseaux,dense_only,dense_only,True,dense_only,True
6,como aprender a programar en rust,dense_only,dense_only,True,dense_only,True
